# 🧠 Lesson 01: Why Ensembles Work

**Time:** ~90 minutes  **Prerequisites:** Lesson 00, basic probability (binomial distribution), variance

### 🎯 Learning goals
By the end of this lesson you will be able to:
1. Explain *why* averaging many models reduces error, using the wisdom-of-crowds intuition **and** a formula.
2. Show with a simulation that **correlation** between models limits how much an ensemble can help.
3. Decompose prediction error into **bias², variance and noise**, and measure each one empirically.
4. State the two ingredients of a good ensemble: **accurate** members that are **diverse**.
5. Place bagging, boosting and stacking on one map, and say which problem each attacks.

> 💡 **Why a whole lesson on theory?** Because in practice you will be asked, "why did you choose a random forest over a
> single model?", and "it scored higher" is a weak answer. By the end of today you'll have a strong one.

In [1]:
import sys; sys.path.append("../utils")          # lets us import the course helpers
from course_utils import *                       # load_housing(), check(), set_style(), ...
import numpy as np, pandas as pd, matplotlib.pyplot as plt

set_style()
np.random.seed(SEED)
from scipy import stats
from scipy.stats import binom, norm
from sklearn.base import clone

## 1 · A story about an ox

In 1906, the statistician **Francis Galton** attended a country fair in England where 787 people paid to guess the
weight of an ox after it had been slaughtered and dressed. Individual guesses were all over the place. Yet Galton found the **median guess was 1,207 lb; the true weight was 1,198 lb**. The crowd was off by less than 1%, better than almost every individual in it.

Why? Each person's error is a mix of *over*-estimates and *under*-estimates. Average many of them and the errors **cancel**.
That's the entire principle. Let's reproduce it with a simulation.

Suppose each guesser is *unbiased but noisy*: guess = truth + random error with standard deviation 75 lb.

In [2]:
rng = np.random.default_rng(SEED)
truth, n_people, sigma = 1198, 787, 75

guesses = truth + rng.normal(0, sigma, n_people)

individual_errors = np.abs(guesses - truth)
crowd_error = abs(guesses.mean() - truth)

print(f"Typical individual error : {individual_errors.mean():6.1f} lb")
print(f"Best individual error    : {individual_errors.min():6.1f} lb   (lucky!)")
print(f"Crowd-average error      : {crowd_error:6.1f} lb")
print(f"People beaten by the crowd: {(individual_errors > crowd_error).mean():.1%}")

Typical individual error :   59.0 lb
Best individual error    :    0.0 lb   (lucky!)
Crowd-average error      :    2.1 lb
People beaten by the crowd: 98.2%


### 🧪 Worked Example 1.1: how fast does the crowd improve?

If errors are independent with standard deviation $\sigma$, the average of $n$ guesses has standard deviation

$$\text{SD}(\bar{x}) = \frac{\sigma}{\sqrt{n}}.$$

With $\sigma = 75$ and $n = 787$ that is $75/\sqrt{787} \approx 2.7$ lb. Let's check the formula against simulation:
we'll repeat the experiment 2,000 times for different crowd sizes.

In [3]:
sizes = [1, 2, 5, 10, 25, 50, 100, 300, 800]
simulated_sd = []
for n in sizes:
    crowd_means = (truth + rng.normal(0, sigma, size=(2000, n))).mean(axis=1)   # 2000 repeated crowds of size n
    simulated_sd.append(crowd_means.std())

plt.figure(figsize=(7, 4))
plt.loglog(sizes, simulated_sd, "o", color=PALETTE["blue"], label="simulated")
plt.loglog(sizes, sigma / np.sqrt(sizes), "-", color=PALETTE["orange"], label=r"theory $\sigma/\sqrt{n}$")
plt.xlabel("crowd size n"); plt.ylabel("std. dev. of the crowd's average (lb)")
plt.title("Averaging shrinks error like 1/√n"); plt.legend(); plt.show()

Beautiful: theory and simulation agree. But notice the catch hiding in that formula: it assumed the errors are **independent**.
Real models trained on the same data, with the same features, are *not* independent: when one is fooled, its siblings tend to be fooled too.
Hold that thought; we'll quantify it in Section 3.

## 2 · Voting: the Condorcet jury theorem

Now take a *classification* problem. Imagine $n$ classifiers, each correct with probability $p > 0.5$, and suppose they make **independent** errors.
The ensemble predicts by **majority vote**. How often is the majority right?

If $X$ is the number of correct voters, $X \sim \text{Binomial}(n, p)$ and the majority is right when $X > n/2$:

$$P(\text{majority correct}) = \sum_{k > n/2} \binom{n}{k} p^k (1-p)^{n-k}.$$

### 🧪 Worked Example 1.2: three 70%-accurate classifiers
The majority is right if **all 3** are right, or **exactly 2** are right:

$$0.7^3 + 3 \cdot 0.7^2 \cdot 0.3 = 0.343 + 0.441 = \mathbf{0.784}.$$

Three mediocre classifiers (70%) become one good one (78.4%): *for free*, if their errors are independent. Verify, then scale up:

In [4]:
def majority_accuracy(n, p):
    """P(majority of n independent voters, each right with prob p, is right).  n should be odd."""
    return binom.sf(n // 2, n, p)          # sf(k) = P(X > k)

print("3 voters at 70%:", round(majority_accuracy(3, 0.70), 4))

ns = np.arange(1, 202, 2)
plt.figure(figsize=(8, 4.2))
for p in [0.51, 0.55, 0.60, 0.70]:
    plt.plot(ns, [majority_accuracy(n, p) for n in ns], label=f"each voter {p:.0%} accurate")
plt.axhline(1, color="black", lw=.6, ls=":")
plt.xlabel("number of independent voters"); plt.ylabel("accuracy of the majority vote")
plt.title("Condorcet: weak voters, strong committee"); plt.legend(); plt.show()

3 voters at 70%: 0.784


🔮 **Predict:** the curve for a barely-better-than-coin-flip voter (51%) rises *slowly*. If every voter was only **49%** accurate
(worse than guessing), what would happen as you add voters? *(Run `majority_accuracy(201, 0.49)` to find out.)*

> ⚠️ **The flip side:** if members are *worse than random*, the committee gets **worse** as it grows. Ensembles amplify
> whatever the members have in common: good or bad. This is why every ensemble method insists on members that beat chance.

## 3 · The catch: correlated errors

Real models see the same data. Let's simulate voters whose mistakes are **correlated**: all voters are nudged by one *shared* random
factor (think "the same confusing example fools them all"). We control the correlation $\rho$ between voters.

In [5]:
def simulate_majority(n_voters, p, rho, n_trials=3000, seed=0):
    """Majority-vote accuracy when each voter is right w.p. p, with pairwise correlation rho in 'being fooled'."""
    rng = np.random.default_rng(seed)
    common = rng.normal(size=(n_trials, 1))                 # the shared 'difficulty of this example'
    own = rng.normal(size=(n_trials, n_voters))             # each voter's private noise
    z = np.sqrt(rho) * common + np.sqrt(1 - rho) * own
    correct = z < norm.ppf(p)                                # right with probability exactly p, but correlated
    return (correct.sum(axis=1) > n_voters / 2).mean()

ns = [1, 3, 5, 9, 15, 25, 41, 71, 101]
plt.figure(figsize=(8, 4.4))
for rho in [0.0, 0.05, 0.2, 0.5, 1.0]:
    plt.plot(ns, [simulate_majority(n, 0.65, rho) for n in ns], "o-", ms=4, label=f"correlation ρ = {rho}")
plt.xlabel("number of voters"); plt.ylabel("majority-vote accuracy")
plt.title("Each voter is 65% accurate. Correlation caps the benefit")
plt.legend(); plt.show()

Study that plot: it's the most important picture of the lesson:

* **ρ = 0** (independent): accuracy climbs toward 100%.
* **ρ = 0.2**: it still helps, but plateaus far below 100%.
* **ρ = 1** (identical voters): a committee of 101 clones is exactly as good as one voter: **65%**. Cloning adds nothing.

### 🧪 Worked Example 1.3: the formula behind the plateau (for *regression*)

For regression we average predictions instead of voting. Let each of $B$ models have prediction variance $\sigma^2$ and pairwise correlation $\rho$. Then

$$\boxed{\;\text{Var}\Big(\frac{1}{B}\sum_{b=1}^{B} \hat f_b\Big) \;=\; \rho\,\sigma^2 \;+\; \frac{1-\rho}{B}\,\sigma^2\;}$$

*Derivation sketch.* The variance of a sum is the sum of all entries in the covariance matrix: $B$ diagonal terms $\sigma^2$ and $B(B-1)$ off-diagonal terms $\rho\sigma^2$. Divide by $B^2$:
$\frac{B\sigma^2 + B(B-1)\rho\sigma^2}{B^2} = \frac{\sigma^2}{B} + \rho\sigma^2 - \frac{\rho\sigma^2}{B}$, which rearranges to the boxed result. ∎

Read the two terms like a doctor reading a scan:

* $\frac{1-\rho}{B}\sigma^2$: **shrinks as you add models**. This is the part ensembling can remove.
* $\rho\,\sigma^2$: **a floor that no number of models can break**. Only *lower correlation* can lower it.

> 💡 **That one formula explains the design of random forests.** Bagging trees gets you $B$ large; but the trees are
> correlated (same data, same features → similar trees), so $\rho$ is large and the floor bites. The random forest's
> extra trick (each split sees only a random subset of features) exists purely to **lower ρ**. We'll watch it work in Lesson 02.

Let's verify the formula numerically with $B=10$, $\rho=0.3$, $\sigma^2 = 1$: theory says $0.3 + 0.7/10 = 0.37$.

In [6]:
def variance_of_average(B, rho, sigma2=1.0):
    return rho * sigma2 + (1 - rho) * sigma2 / B

B, rho = 10, 0.3
cov = (1 - rho) * np.eye(B) + rho * np.ones((B, B))             # variance 1 on the diagonal, rho off it
draws = np.random.default_rng(0).multivariate_normal(np.zeros(B), cov, size=200_000)
print(f"theory    : {variance_of_average(B, rho):.4f}")
print(f"simulation: {draws.mean(axis=1).var():.4f}")

theory    : 0.3700
simulation: 0.3699


In [7]:
Bs = np.unique(np.logspace(0, 3, 40).astype(int))
plt.figure(figsize=(8, 4.2))
for rho in [0.0, 0.1, 0.3, 0.6, 1.0]:
    plt.semilogx(Bs, [variance_of_average(b, rho) for b in Bs], label=f"ρ = {rho}")
plt.xlabel("number of models B (log scale)"); plt.ylabel("variance of the ensemble average (σ²=1)")
plt.title("The correlation floor: more models can't go below ρσ²")
plt.legend(); plt.show()

## 4 · Bias, variance, noise: measuring them for real

The averaging story above is about **variance**. To see the whole picture we need the classic decomposition.
For a new point $x$ with true function $f(x)$ and noisy observation $y = f(x) + \varepsilon$ (noise variance $\sigma_\varepsilon^2$), a model $\hat f$ trained on a random training set has expected squared error

$$\mathbb{E}\big[(y - \hat f(x))^2\big] \;=\; \underbrace{\big(\mathbb{E}[\hat f(x)] - f(x)\big)^2}_{\text{Bias}^2} \;+\; \underbrace{\mathbb{E}\big[(\hat f(x) - \mathbb{E}[\hat f(x)])^2\big]}_{\text{Variance}} \;+\; \underbrace{\sigma_\varepsilon^2}_{\text{irreducible noise}}.$$

* **Bias²**: on average (over many training sets) how far is the model from the truth? *Too-simple models are biased.*
* **Variance**: how much does the model change when the training set changes? *Too-flexible models are jumpy.*
* **Noise**: randomness in $y$ itself. Nobody can predict that. It's the floor for *every* model.

We can't measure these on real data because we don't know $f$. But in a simulation we do! Our experiment:

1. True function: $f(x)=\sin(2\pi x)$ on $[0,1]$, noise SD = 0.3.
2. Generate **150 different training sets** of 100 points each.
3. Fit the same model type on each; look at the 150 fitted curves.

First, a picture. 🔮 *Which of the three panels will have the most 'spaghetti'?*

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, GradientBoostingRegressor

def true_function(x):
    return np.sin(2 * np.pi * x)

def bias_variance(model, n_train=100, n_rep=150, noise=0.3, seed=0, return_preds=False):
    """Fit `model` on n_rep independent training sets; estimate bias², variance, noise and total error."""
    rng = np.random.default_rng(seed)
    x_grid = np.linspace(0, 1, 200)
    preds = np.empty((n_rep, x_grid.size))
    for r in range(n_rep):
        x = rng.uniform(0, 1, n_train)
        y = true_function(x) + rng.normal(0, noise, n_train)
        preds[r] = clone(model).fit(x[:, None], y).predict(x_grid[:, None])
    bias2 = np.mean((preds.mean(axis=0) - true_function(x_grid)) ** 2)
    variance = np.mean(preds.var(axis=0))
    y_new = true_function(x_grid) + rng.normal(0, noise, preds.shape)      # fresh noisy targets
    out = {"bias2": bias2, "variance": variance, "noise": noise ** 2,
           "bias2+var+noise": bias2 + variance + noise ** 2, "measured_mse": np.mean((y_new - preds) ** 2)}
    return (out, x_grid, preds) if return_preds else out

panel_models = {
    "Shallow tree (depth 2)": DecisionTreeRegressor(max_depth=2),
    "Deep tree (no limit)": DecisionTreeRegressor(),
    "Bagged deep trees (50)": BaggingRegressor(DecisionTreeRegressor(), n_estimators=50, random_state=0),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, model) in zip(axes, panel_models.items()):
    _, xg, preds = bias_variance(model, n_rep=30, return_preds=True)
    for p in preds[:20]:
        ax.plot(xg, p, color=PALETTE["blue"], alpha=.25, lw=1)
    ax.plot(xg, preds.mean(axis=0), color=PALETTE["orange"], lw=2.5, label="average fit")
    ax.plot(xg, true_function(xg), "k--", lw=1.5, label="truth")
    ax.set_title(name); ax.set_ylim(-1.8, 1.8)
axes[0].legend(loc="lower left")
plt.tight_layout(); plt.show()

Read the panels left to right:

* **Shallow tree**: the 20 fits hug each other (low variance) but the *average* is a staircase far from the sine wave (high bias).
* **Deep tree**: the average fit lands on the truth (low bias) but individual fits are a chaotic mess (high variance).
* **Bagged deep trees**: still low bias, but the spaghetti has been *combed* into a tight bundle. We kept the flexibility and removed the jitter. **That is bagging.**

Now let's put numbers on all of it, adding a linear model (very biased) and gradient boosting (a preview of Lesson 03).

In [9]:
zoo = {
    "Linear regression": LinearRegression(),
    "Tree depth 2": DecisionTreeRegressor(max_depth=2),
    "Tree depth 6": DecisionTreeRegressor(max_depth=6),
    "Deep tree (no limit)": DecisionTreeRegressor(),
    "Bagged deep trees (50)": BaggingRegressor(DecisionTreeRegressor(), n_estimators=50, random_state=0),
    "Gradient boosting (100 × depth 2)": GradientBoostingRegressor(n_estimators=100, max_depth=2, learning_rate=0.1, random_state=0),
}
bv = pd.DataFrame({name: bias_variance(m) for name, m in zoo.items()}).T
bv.round(4)

,bias2,variance,noise,bias2+var+noise,measured_mse
Linear regression,0.1999,0.0064,0.09,0.2963,0.2967
Tree depth 2,0.0300,0.0437,0.09,0.1638,0.1639
Tree depth 6,0.0005,0.0596,0.09,0.1501,0.1515
Deep tree (no limit),0.0007,0.0885,0.09,0.1792,0.1807
Bagged deep trees (50),0.0004,0.0423,0.09,0.1328,0.1340
Gradient boosting (100 × depth 2),0.0006,0.0276,0.09,0.1183,0.1197


In [10]:
fig, ax = plt.subplots(figsize=(9, 4.4))
bottom = np.zeros(len(bv))
for col, color in [("noise", PALETTE["gray"]), ("bias2", PALETTE["red"]), ("variance", PALETTE["blue"])]:
    ax.barh(bv.index, bv[col], left=bottom, label=col.replace("bias2", "bias²"), color=color)
    bottom += bv[col].values
ax.invert_yaxis(); ax.set_xlabel("expected squared error  =  noise + bias² + variance"); ax.legend(loc="lower right")
ax.set_title("Where does each model's error come from?"); plt.tight_layout(); plt.show()

### 🔎 Reading the table

| model | bias² | variance | story |
|---|---|---|---|
| Linear | **large** | tiny | a straight line can't follow a sine → *underfits* |
| Tree depth 2 | moderate | small | too coarse: the staircase has only 4 steps |
| Deep tree | ≈ 0 | **≈ noise level** | it chases the noise in each training set → *overfits* |
| **Bagged deep trees** | ≈ 0 | **less than half** of the deep tree's | same low bias, variance averaged away |
| **Gradient boosting** | very small | small | many small corrections reduce *bias* while a small learning rate keeps variance in check |

Two more observations to savour:

1. The column **`bias2+var+noise`** matches **`measured_mse`** closely: the decomposition isn't a metaphor, it's arithmetic.
2. Every model's total error is **≥ noise² = 0.09**. That's the irreducible floor. If your model's error is near the noise floor, stop tuning and go home.

> 💡 **Rule of thumb.** *Bagging attacks variance. Boosting attacks bias.* Keep that sentence in your pocket: we'll use it to choose
> methods all fortnight.

## 5 · The map of ensemble methods

```
                         ┌──────────────────────────────────────────────────────────┐
                         │                    ENSEMBLE LEARNING                      │
                         └──────────────────────────────────────────────────────────┘
        PARALLEL (independent members)             SEQUENTIAL (each learns from the last)        LEARNED COMBINATION
   ┌───────────────────────────────────┐        ┌───────────────────────────────────┐      ┌─────────────────────────┐
   │  BAGGING  (Lesson 02)             │        │  BOOSTING (Lessons 03–04)         │      │  STACKING (Lesson 05)   │
   │  bootstrap samples → many models  │        │  reweight/residual fitting        │      │  a "meta-model" learns  │
   │  → average / vote                 │        │  → weighted sum of small models   │      │  how to blend *different│
   │  ➜ cuts VARIANCE                  │        │  ➜ cuts BIAS (and some variance)  │      │  kinds* of models       │
   │  e.g. Random Forest               │        │  e.g. AdaBoost, GBM, XGBoost,     │      │  VOTING = simple, fixed │
   │                                   │        │       LightGBM, CatBoost          │      │  version of this idea   │
   └───────────────────────────────────┘        └───────────────────────────────────┘      └─────────────────────────┘
```

## 6 · The two ingredients, in real data

Everything so far gives us the recipe for a good committee:

1. **Accurate members**: each one clearly better than chance (or than the baseline).
2. **Diverse members**: their *errors* are as uncorrelated as possible.

Let's check this on the housing data. We'll train three very different models, then look at how correlated their **residuals** (errors) are.

In [11]:
X_train, X_test, y_train, y_test = get_housing_split()
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline

members = {
    "Linear":       LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
    "GradBoost":    GradientBoostingRegressor(random_state=SEED),
}
preds = {}
for name, model in members.items():
    preds[name] = make_pipeline(make_preprocessor(X_train), model).fit(X_train, y_train).predict(X_test)

residuals = pd.DataFrame({name: y_test.values - p for name, p in preds.items()})
print("Correlation of test-set errors between models:")
residuals.corr().round(2)

Correlation of test-set errors between models:


,Linear,RandomForest,GradBoost
Linear,1.00,0.50,0.55
RandomForest,0.50,1.00,0.86
GradBoost,0.55,0.86,1.00


The linear model's errors are *less* correlated with the tree models' errors than the two tree ensembles are with each other:
it makes **different kinds of mistakes** (it can't bend; they can). That's diversity. Does it pay off? Let's average every possible subset of the three
(a first, tiny *voting regressor*) and see:

In [12]:
from itertools import combinations

rows = []
for k in (1, 2, 3):
    for combo in combinations(preds, k):
        avg = np.mean([preds[c] for c in combo], axis=0)
        rows.append({"committee": " + ".join(combo), "size": k, **regression_report(y_test, avg)})
committees = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
committees.round(3)

,committee,size,RMSE,MAE,R2,MAPE_%
0,GradBoost,1,108.249,64.830,0.908,15.049
1,Linear + RandomForest + GradBoost,3,109.570,69.060,0.905,17.941
2,RandomForest + GradBoost,2,111.417,68.353,0.902,15.662
3,Linear + GradBoost,2,114.067,72.796,0.898,21.576
4,Linear + RandomForest,2,117.924,75.279,0.891,20.830
5,RandomForest,1,122.808,78.631,0.881,18.806
6,Linear,1,149.916,100.460,0.823,33.816


### 🔎 What do we learn?

(Numbers below are from our fixed-seed run; yours should match closely.)

* **Insurance value.** The average of all three models lands within about 1% of the best single model and far ahead of the weakest, and you didn't need to know in advance which member would be best.
* **Equal votes dilute a dominant member.** Gradient Boosting alone is *still* the best row in the table: when one member is clearly better than the others, giving every member an equal vote drags the result toward the weaker ones. Ensembles are a tool, not a religion.
* **Diversity can pay.** Adding the linear model (whose errors are only ~0.5 correlated with the trees') to the RF + GB pair *improved* it, even though the linear model alone is the worst performer. That is the "diverse" half of the recipe at work.
* In **Lesson 05** we'll fix the dilution problem by *learning* the weights (stacking), and **Lesson 06** teaches you to test claims like "A beats B" rigorously instead of eyeballing one split.

### ✋ Check your understanding

Try to answer *out loud* before you open the answer. If you can't say it simply, you don't own it yet.

**Q1. Your 100-tree forest has per-tree variance σ² = 4 and average pairwise correlation ρ = 0.5. What is the variance of the forest's prediction? What's the best you could ever get by adding more trees?**

<details><summary>Show answer</summary>

Variance $= \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2 = 0.5\cdot4 + \frac{0.5}{100}\cdot 4 = 2 + 0.02 = 2.02$. With infinitely many trees you approach the floor $\rho\sigma^2 = \mathbf{2.0}$: you'd have removed only 1% more. To go lower you must **reduce the correlation** (e.g. random feature subsets) or reduce σ².

</details>

**Q2. Why does bagging help a deep decision tree a lot but help linear regression very little?**

<details><summary>Show answer</summary>

Deep trees have **high variance** (and low bias), so averaging removes the dominant error source. Linear regression is already **low variance / high bias**; averaging many nearly identical linear fits just returns nearly the same line. Bagging does not fix bias.

</details>

**Q3. Three classifiers each have 60% accuracy but make *exactly the same mistakes*. What is the accuracy of their majority vote?**

<details><summary>Show answer</summary>

**60%.** Identical mistakes mean ρ = 1: the vote simply equals any one member. Diversity is the whole point.

</details>

**Q4. Why can't a model's error ever drop below σ²_noise?**

<details><summary>Show answer</summary>

That part of $y$ is random: it isn't a function of the features. A perfect model of $f(x)$ still can't predict the random wiggle $\varepsilon$. If you're near this floor, you're done; additional tuning is chasing noise (and will overfit).

</details>

## 7 · Problems

Four problems, rising in difficulty. Everything you need was defined above (`majority_accuracy`, `variance_of_average`,
`bias_variance`, `preds`, …).

### 🧩 Problem 01.1 ⭐: Exact majority vote

Eleven independent classifiers are each **55%** accurate. Using `majority_accuracy`, compute the accuracy of their majority vote and store it in `p_majority`.
Also compute the accuracy when you have **101** such voters and store it in `p_majority_101`.

<details><summary>💡 Hint 1</summary>

The function signature is `majority_accuracy(n, p)`.

</details>

<details><summary>💡 Hint 2</summary>

Just call it twice: `majority_accuracy(11, 0.55)` and `majority_accuracy(101, 0.55)`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
p_majority = TODO       # YOUR CODE HERE
p_majority_101 = TODO   # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("01.1", lambda: abs(p_majority - 0.6331) < 0.001 and p_majority_101 > 0.8,
      "11 voters at 55% should give ≈ 0.633, and 101 voters should be above 0.8.", needs=[p_majority, p_majority_101])

### 🧩 Problem 01.2 ⭐⭐: Simulate the correlation floor

Simulate **B = 25** predictors, each with standard deviation σ = 2 (variance 4) and pairwise correlation ρ = 0.4.

1. Build the covariance matrix `cov` (4 on the diagonal, `0.4 * 4 = 1.6` off the diagonal).
2. Draw 100,000 samples with `rng.multivariate_normal` and compute the variance of the **average** of the 25 predictors → `sim_var`.
3. Compute the theoretical value from the formula → `theory_var`.

The two numbers should agree within a few percent.

<details><summary>💡 Hint 1</summary>

The covariance matrix is `sigma2 * ((1 - rho) * np.eye(B) + rho * np.ones((B, B)))`.

</details>

<details><summary>💡 Hint 2</summary>

`draws = rng.multivariate_normal(np.zeros(B), cov, size=100_000)` has shape (100000, 25). The average of the predictors is `draws.mean(axis=1)`.

</details>

<details><summary>💡 Hint 3</summary>

`theory_var = variance_of_average(B, rho, sigma2)`: it was defined in Section 3.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
rng = np.random.default_rng(1)
B, rho, sigma2 = 25, 0.4, 4.0

sim_var = TODO      # YOUR CODE HERE
theory_var = TODO   # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("01.2", lambda: abs(theory_var - 1.696) < 1e-6 and abs(sim_var - theory_var) / theory_var < 0.05,
      "Theory: 0.4*4 + 0.6*4/25 = 1.696.  Your simulation should be within 5% of it.", needs=[sim_var, theory_var])

### 🧩 Problem 01.3 ⭐⭐: Bias or variance?

Run `bias_variance` for a `DecisionTreeRegressor(max_depth=4)` and store the result dict in `bv4`.
Then decide: is this model's error dominated by **bias** or by **variance**? Store the string `"bias"` or `"variance"` in `dominant`.

*Reflection (write 2 sentences in a markdown cell below):* would you expect bagging to help this model a lot or only a little? Why? (Hint: which term of the decomposition can bagging shrink?)

<details><summary>💡 Hint 1</summary>

`bv4 = bias_variance(DecisionTreeRegressor(max_depth=4))` returns a dict with keys `bias2` and `variance`.

</details>

<details><summary>💡 Hint 2</summary>

Compare `bv4['bias2']` with `bv4['variance']`: whichever is larger dominates.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
bv4 = TODO         # YOUR CODE HERE
dominant = TODO    # YOUR CODE HERE  -> "bias" or "variance"

# ---- self-check (don't edit) ----
check("01.3", lambda: dominant == ("bias" if bv4["bias2"] > bv4["variance"] else "variance") and abs(bv4["bias2+var+noise"] - bv4["measured_mse"]) < 0.02,
      "Make sure `bv4` is the dict returned by bias_variance, and that `dominant` matches its larger entry.", needs=[bv4, dominant])

### 🧩 Problem 01.4 ⭐⭐⭐: Find the best committee

Using the housing predictions in the dict `preds` (keys: `"Linear"`, `"RandomForest"`, `"GradBoost"`) and `y_test`:

* try **every non-empty subset** of the three models, averaging their predictions with equal weights,
* store the winning subset as a tuple of names in `best_subset` (e.g. `("RandomForest", "GradBoost")`),
* and its test RMSE in `best_rmse`.

Then answer in a markdown cell: *is the winner an ensemble, or a single model? Why do you think so?*

<details><summary>💡 Hint 1</summary>

`itertools.combinations(names, k)` generates all subsets of size k. Loop over k = 1, 2, 3.

</details>

<details><summary>💡 Hint 2</summary>

For each subset: `avg = np.mean([preds[n] for n in subset], axis=0)`, then RMSE `np.sqrt(np.mean((y_test - avg) ** 2))`.

</details>

<details><summary>💡 Hint 3</summary>

Keep track of the smallest RMSE seen so far (or build a list of (rmse, subset) and take `min`).

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
from itertools import combinations

best_subset = TODO   # YOUR CODE HERE  (a tuple of model names)
best_rmse = TODO     # YOUR CODE HERE

# ---- self-check (don't edit) ----
def _expected():
    from itertools import combinations
    return min((np.sqrt(np.mean((y_test.values - np.mean([preds[n] for n in s], axis=0)) ** 2)), s)
               for k in (1, 2, 3) for s in combinations(preds, k))
check("01.4", lambda: abs(best_rmse - _expected()[0]) < 1e-6 and tuple(sorted(best_subset)) == tuple(sorted(_expected()[1])),
      "Try all 7 subsets and keep the one with the lowest RMSE.", needs=[best_subset, best_rmse])

In [17]:
scoreboard()

Scoreboard: 4 solved, 0 to fix, 0 not attempted
   ✅ 01.1
   ✅ 01.2
   ✅ 01.3
   ✅ 01.4


## 🏁 Recap

* **Averaging cancels independent errors** (SD shrinks like $1/\sqrt n$); majority voting makes weak classifiers strong (Condorcet).
* **Correlation is the enemy:** $\text{Var}=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2$: the first term is a floor only diversity can lower.
* Error = **bias² + variance + noise**. *Bagging attacks variance; boosting attacks bias.*
* A good ensemble needs members that are **accurate** and **diverse**.

### ➡️ Next up: **Lesson 02: Bagging & Random Forest**
We'll build bagging *from scratch* in ~15 lines, discover the free validation set hiding inside every bootstrap (out-of-bag),
and meet the random forest's secret weapon for cutting correlation.

*Think about it tonight:* if every tree in a forest is trained on the same 2,400 houses… how do we get 300 *different* trees out of them?